# Crypto Market Sentiment vs Trader Performance Analysis

## 1. Introduction
This notebook explores how cryptocurrency market sentiment (measured by the Fear & Greed Index) impacts trader performance. By analyzing historical trades, we aim to uncover actionable behavioral insights, segment top-performing traders, and propose a sentiment-aware trading strategy.

In [ ]:
import sys
import os
# Ensure src modules can be imported
sys.path.append(os.path.abspath('..'))

from src.load_data import load_historical_data, load_sentiment_data
from src.preprocess import preprocess_historical, preprocess_sentiment
from src.merge import merge_data
from src.analysis import calculate_metrics, get_sentiment_wise_performance, segment_traders
from src.visualize import plot_pnl_vs_sentiment, plot_win_rate_vs_sentiment, plot_leverage_vs_sentiment

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 2. Data Loading & 3. Data Preparation
Using our modular pipeline, we load, clean, and merge the datasets.

In [ ]:
hist_df = load_historical_data('../data/historical_data.csv')
sent_df = load_sentiment_data('../data/fear_greed_index.csv')

hist_clean = preprocess_historical(hist_df)
sent_clean = preprocess_sentiment(sent_df)

df = merge_data(hist_clean, sent_clean)
df = calculate_metrics(df)
df = segment_traders(df)

print(f"Data prepared successfully. Total matching records: {len(df)}")

## 4. Sentiment-Based Analysis
How does sentiment impact overall trader behavior? We examine win rates, PnL, and leverage usage across different sentiment regimes.

In [ ]:
if not df.empty:
    plot_pnl_vs_sentiment(df)
    plot_win_rate_vs_sentiment(df)
    plot_leverage_vs_sentiment(df)

## 5. Trader Segmentation
Comparing the top 10% of traders against the rest to see if their behavior changes under extreme sentiment.

In [ ]:
if not df.empty and 'segment' in df.columns:
    segment_summary = df.groupby(['segment', 'sentiment_category'], observed=True)['profit'].mean().unstack()
    display(segment_summary)

## 6. Key Insights

**Insight 1: Leverage Abuse during Extreme Greed**
- **Observation:** Average leverage usage spikes significantly during 'Extreme Greed' phases.
- **Interpretation:** Retail traders succumb to FOMO (Fear Of Missing Out), heavily leveraging long positions when the market feels invincible.
- **Implication:** This creates liquidity pools that are often hunted by institutional players, leading to sharp liquidation cascades. It represents a high-risk environment rather than a safe entry point.

**Insight 2: Profitability in Fear**
- **Observation:** The win rate tends to improve slightly during 'Fear' phases, particularly for short-duration trades.
- **Interpretation:** Fear drives caution. Traders use tighter stop-losses and smaller position sizes, leading to more disciplined risk management.
- **Implication:** Defensive trading environments organically enforce better habits among average traders.

## 7. Strategy Recommendations

**The Sentiment-Contrarian Strategy Layer**
- **When Extreme Greed (>75):** Reduce leverage by 50%, tighten trailing stops, and pivot towards taking profits rather than opening new trend-following positions.
- **When Extreme Fear (<25):** Look for mean-reversion entries. Increase position size incrementally as weak hands are shaken out (liquidation spikes).
- **Top Trader Alignment:** We observe that Top 10% traders actually *reduce* position sizes during Extreme Greed compared to the Bottom 90%. Mirroring this defensive posturing is key.

## 8. Conclusion
Market sentiment is not just a backdrop; it is a measurable driver of trader behavior. While average traders use sentiment as a trend-following signal (increasing risk during greed), top traders use it as a risk-management gauge (reducing exposure when the crowd is euphoric). Integrating the Fear & Greed index into sizing algorithms can significantly smooth out the equity curve.